# Round 3: relaxed judge on Split CIFAR-10

Self-contained copy of the `continual_learning` project. Run all cells in order.

**Before running:** turn on a GPU.
- **Kaggle:** Settings → Accelerator → GPU T4 x2 (or P100), and Settings → Internet → On (needed to download CIFAR-10).
- **Colab:** Runtime → Change runtime type → T4 GPU.

The run (6 judge settings tuned on val with 2 seeds each, then the winner on test with
3 seeds) takes roughly 1.5-2 hours on a T4. Progress is printed as it goes and also written to
`results/suite_progress.log`. Tip: use Save Version -> Save & Run All (Commit) so it keeps running
with the browser closed.

In [ ]:
import os, torch
os.makedirs("/tmp/cl_project/cl", exist_ok=True)
os.chdir("/tmp/cl_project")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Turn one on (see the first cell) or this will take many hours.")

## Project files
Each cell below writes one file of the project.

In [ ]:
%%writefile cl/__init__.py
# package marker

In [ ]:
%%writefile cl/data.py
"""Split MNIST / Split CIFAR-10: a 10-class dataset cut into sequential tasks of
disjoint class pairs (task 0 = classes 0-1, task 1 = classes 2-3, ...).

Each task carries four splits:
  - train: what the learner sees while that task is "current"
  - judge: the first `judge_per_task` samples of the held-out slice (default 40).
           Only for in-loop decisions of a strategy (the judge / rollback in our
           method). Counted against that strategy's memory budget when stored.
  - val:   the rest of the held-out slice of the *training* set (never trained on).
           Used for hyperparameter / design selection (--eval-split val); disjoint
           from judge, so tuning never reuses the data a strategy makes decisions on.
  - test:  the official test set for that task's classes; used only for reporting.

Everything is loaded into memory as float tensors, which is much faster than a
DataLoader for models this small. MNIST is flattened to 784 (for the MLP); CIFAR-10
stays (3, 32, 32) and is normalised per channel. No data augmentation.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch
from torchvision import datasets

N_CLASSES = 10
DATASETS = ("mnist", "cifar10")

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


@dataclass
class TaskData:
    task_id: int
    classes: tuple[int, ...]
    x_train: torch.Tensor
    y_train: torch.Tensor
    x_val: torch.Tensor
    y_val: torch.Tensor
    x_test: torch.Tensor
    y_test: torch.Tensor
    x_judge: torch.Tensor | None = None
    y_judge: torch.Tensor | None = None

    def __repr__(self) -> str:
        n_judge = 0 if self.y_judge is None else len(self.y_judge)
        return (f"Task {self.task_id} classes={self.classes} train={len(self.y_train)} "
                f"judge={n_judge} val={len(self.y_val)} test={len(self.y_test)}")


def _load_mnist(root: str, train: bool) -> tuple[torch.Tensor, torch.Tensor]:
    ds = datasets.MNIST(root, train=train, download=True)
    x = ds.data.float().div_(255).sub_(MNIST_MEAN).div_(MNIST_STD).view(-1, 28 * 28)
    return x, ds.targets.clone()


def _load_cifar10(root: str, train: bool) -> tuple[torch.Tensor, torch.Tensor]:
    ds = datasets.CIFAR10(root, train=train, download=True)
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float().div_(255)  # N,3,32,32
    x.sub_(torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)).div_(torch.tensor(CIFAR_STD).view(1, 3, 1, 1))
    return x, torch.tensor(np.array(ds.targets), dtype=torch.long)


_LOADERS = {"mnist": _load_mnist, "cifar10": _load_cifar10}


def make_split(dataset: str = "mnist", root: str = "data", n_tasks: int = 5, val_fraction: float = 0.1,
               seed: int = 0, judge_per_task: int = 40) -> list[TaskData]:
    """Return `n_tasks` tasks; task t holds classes [t*k, ..., t*k + k-1], k = 10 / n_tasks.

    The judge set is carved from the held-out (val) slice, not from train, so the
    train split -- and hence every test number -- is unaffected by it."""
    if N_CLASSES % n_tasks:
        raise ValueError(f"n_tasks must divide {N_CLASSES}, got {n_tasks}")
    load = _LOADERS[dataset]
    k = N_CLASSES // n_tasks
    x_tr, y_tr = load(root, train=True)
    x_te, y_te = load(root, train=False)
    gen = torch.Generator().manual_seed(seed)

    tasks = []
    for t in range(n_tasks):
        classes = tuple(range(t * k, (t + 1) * k))
        cls = torch.tensor(classes)

        m = torch.isin(y_tr, cls)
        x, y = x_tr[m], y_tr[m]
        perm = torch.randperm(len(y), generator=gen)
        n_val = int(len(y) * val_fraction)
        val_idx, tr_idx = perm[:n_val], perm[n_val:]
        if not 0 <= judge_per_task < n_val:
            raise ValueError(f"judge_per_task must be in [0, {n_val}), got {judge_per_task}")
        judge_idx, val_idx = val_idx[:judge_per_task], val_idx[judge_per_task:]

        mt = torch.isin(y_te, cls)
        tasks.append(TaskData(t, classes, x[tr_idx], y[tr_idx], x[val_idx], y[val_idx],
                              x_te[mt], y_te[mt], x[judge_idx], y[judge_idx]))
    return tasks


def make_split_mnist(root: str = "data", n_tasks: int = 5, val_fraction: float = 0.1,
                     seed: int = 0, judge_per_task: int = 40) -> list[TaskData]:
    return make_split("mnist", root, n_tasks, val_fraction, seed, judge_per_task)

In [ ]:
%%writefile cl/model.py
"""The learners.

MLP           Split MNIST backbone: 2 hidden layers x 400 units (~480k params).
ReducedResNet18
              Split CIFAR-10 backbone used across the continual-learning literature
              (GEM, A-GEM, MIR, ER-ACE...): ResNet-18 with base width nf=20 instead of 64
              (~1.1M params), so it trains on a laptop.

Both have a single 10-way output head shared across all tasks. How that head is
used (all 10 logits vs. only the current task's) is decided by the scenario in
`cl.metrics.active_logits`, not by the model.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn


class MLP(nn.Module):
    def __init__(self, in_dim: int = 784, hidden: int = 400, n_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class _BasicBlock(nn.Module):
    def __init__(self, cin: int, cout: int, stride: int):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ReducedResNet18(nn.Module):
    def __init__(self, n_classes: int = 10, nf: int = 20):
        super().__init__()
        self.conv1 = nn.Conv2d(3, nf, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(nf)
        layers, cin = [], nf
        for i, stride in enumerate((1, 2, 2, 2)):
            cout = nf * 2 ** i
            layers += [_BasicBlock(cin, cout, stride), _BasicBlock(cout, cout, 1)]
            cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)

In [ ]:
%%writefile cl/metrics.py
"""Evaluation and continual-learning metrics.

Accuracy matrix R: R[i, j] = test accuracy on task j right after training on
task i. Only j <= i is filled (seen tasks); the rest is NaN.

  average accuracy after task i  = mean(R[i, :i+1])
  final average accuracy (ACC)   = mean(R[T-1, :])
  forgetting (Chaudhry et al. 2018), for each old task j < T-1:
      f_j = max_{l in [j, T-2]} R[l, j] - R[T-1, j]
  i.e. the drop from the best accuracy ever reached on j to its final accuracy.
  Reported forgetting is the mean of f_j.
"""
from __future__ import annotations

import numpy as np
import torch

SCENARIOS = ("class", "task")


def active_logits(logits: torch.Tensor, classes: tuple[int, ...] | None, scenario: str) -> torch.Tensor:
    """Class-incremental: all 10 logits compete (no task identity at test time).
    Task-incremental: logits outside the given task's classes are masked out,
    so the model only has to pick between that task's digits."""
    if scenario == "class" or classes is None:
        return logits
    mask = torch.full_like(logits, float("-inf"))
    mask[:, list(classes)] = 0.0
    return logits + mask


@torch.no_grad()
def evaluate(model: torch.nn.Module, x: torch.Tensor, y: torch.Tensor, device: torch.device,
             classes: tuple[int, ...] | None = None, scenario: str = "class",
             batch_size: int = 2000) -> float:
    was_training = model.training
    model.eval()
    correct = 0
    for i in range(0, len(y), batch_size):
        xb, yb = x[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
        pred = active_logits(model(xb), classes, scenario).argmax(1)
        correct += (pred == yb).sum().item()
    model.train(was_training)
    return correct / len(y)


def new_acc_matrix(n_tasks: int) -> np.ndarray:
    return np.full((n_tasks, n_tasks), np.nan)


def avg_acc_curve(R: np.ndarray) -> np.ndarray:
    """Average accuracy over seen tasks after each task."""
    return np.array([np.nanmean(R[i, :i + 1]) for i in range(len(R))])


def final_avg_acc(R: np.ndarray) -> float:
    return float(np.mean(R[-1]))


def forgetting(R: np.ndarray) -> float:
    T = len(R)
    if T < 2:
        return 0.0
    drops = [np.nanmax(R[j:T - 1, j]) - R[T - 1, j] for j in range(T - 1)]
    return float(np.mean(drops))


def format_matrix(R: np.ndarray) -> str:
    T = len(R)
    head = "after\\eval " + " ".join(f"  T{j}  " for j in range(T)) + "  | avg"
    lines = [head]
    for i in range(T):
        cells = " ".join("   -  " if np.isnan(v) else f"{v * 100:5.1f} " for v in R[i])
        lines.append(f"  task {i}   {cells} | {np.nanmean(R[i, :i + 1]) * 100:5.1f}")
    return "\n".join(lines)

In [ ]:
%%writefile cl/strategies.py
"""Continual-learning strategies.

`Strategy` owns the model and the training loop over one task. Subclasses
change behaviour through hooks rather than rewriting the loop:

  before_task(task)            e.g. build a fresh optimizer
  make_batch(x, y, task)       e.g. append replayed samples to the batch
  extra_loss(x, y, task)       e.g. an EWC penalty
  after_step(x, y, task)       e.g. update a memory buffer (gets the *un-augmented* batch)
  after_task(task)             e.g. compute Fisher information

Implemented:
  Naive   plain sequential fine-tuning (forgetting lower bound)
  Replay  experience replay from a fixed-size reservoir-sampled buffer
  EWC     online Elastic Weight Consolidation (diagonal Fisher penalty)
  DERPP   DER++: replay of stored labels *and* stored logits (Buzzega et al. 2020)
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch.func import functional_call, grad, vmap

from .data import N_CLASSES, TaskData


@dataclass
class TrainConfig:
    epochs: int = 2
    batch_size: int = 64
    lr: float = 0.01
    momentum: float = 0.9
    scenario: str = "class"  # "class" or "task" (see cl.metrics.active_logits)


class Strategy:
    name = "base"

    def __init__(self, model: torch.nn.Module, cfg: TrainConfig, device: torch.device, seed: int = 0,
                 input_shape: tuple[int, ...] = (784,), **hparams):
        self.model = model.to(device)
        self.input_shape = tuple(input_shape)  # shape of one sample, for memory buffers
        self.cfg = cfg
        self.device = device
        self.gen = torch.Generator().manual_seed(seed)  # shuffling/sampling, independent of model init
        self.optimizer: torch.optim.Optimizer | None = None
        self.seen_tasks: list[TaskData] = []
        # same_task[c] = bool mask over the classes in c's task (task-IL masking per sample).
        self.same_task = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.bool, device=device)
        self.stats: dict = {"grad_steps": 0}  # saved per seed by run_experiment.py

    # ---- helpers --------------------------------------------------------
    def masked_logits(self, logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Task-IL: each sample only competes among its own task's classes.
        Class-IL: unchanged. Per-sample (not per-batch) so replayed old samples are masked correctly."""
        if self.cfg.scenario == "class":
            return logits
        return logits.masked_fill(~self.same_task[y], float("-inf"))

    def task_loss(self, x: torch.Tensor, y: torch.Tensor, task: TaskData) -> torch.Tensor:
        return F.cross_entropy(self.masked_logits(self.model(x), y), y)

    # ---- hooks ----------------------------------------------------------
    def before_task(self, task: TaskData) -> None:
        cls = list(task.classes)
        for c in cls:
            self.same_task[c, cls] = True
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=self.cfg.lr,
                                         momentum=self.cfg.momentum)

    def make_batch(self, x, y, task: TaskData):
        return x, y

    def extra_loss(self, x, y, task: TaskData) -> torch.Tensor | float:
        return 0.0

    def after_step(self, x, y, task: TaskData) -> None:
        pass

    def after_task(self, task: TaskData) -> None:
        pass

    # ---- core loop ------------------------------------------------------
    def train_task(self, task: TaskData) -> None:
        self.before_task(task)
        self.model.train()
        n = len(task.y_train)
        for self.epoch in range(self.cfg.epochs):
            perm = torch.randperm(n, generator=self.gen)
            for i in range(0, n, self.cfg.batch_size):
                idx = perm[i:i + self.cfg.batch_size]
                x = task.x_train[idx].to(self.device)
                y = task.y_train[idx].to(self.device)
                xb, yb = self.make_batch(x, y, task)
                loss = self.task_loss(xb, yb, task) + self.extra_loss(xb, yb, task)
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                self.optimizer.step()
                self.stats["grad_steps"] += 1
                self.after_step(x, y, task)
        self.after_task(task)
        self.seen_tasks.append(task)


class Naive(Strategy):
    """Fine-tune on each task in turn with no protection against forgetting."""
    name = "naive"


class Replay(Strategy):
    """Experience replay (ER) with a reservoir-sampled buffer.

    The buffer holds at most `mem_size` samples in total. Reservoir sampling keeps
    a uniform random sample of everything seen so far, so old tasks keep a share of
    the buffer without needing task boundaries. Each training step concatenates the
    current batch with an equally sized batch drawn uniformly from the buffer.
    """
    name = "replay"

    def __init__(self, *args, mem_size: int = 500, **kw):
        super().__init__(*args, **kw)
        self.mem_size = mem_size
        self.buf_x = torch.empty(mem_size, *self.input_shape, device=self.device)
        self.buf_y = torch.empty(mem_size, dtype=torch.long, device=self.device)
        self.n_filled = 0
        self.n_seen = 0  # stream count for reservoir sampling

    def sample_buffer(self, n: int) -> torch.Tensor:
        return torch.randint(self.n_filled, (n,), generator=self.gen).to(self.device)

    def make_batch(self, x, y, task):
        if self.n_filled == 0:
            return x, y
        idx = self.sample_buffer(len(y))
        return torch.cat([x, self.buf_x[idx]]), torch.cat([y, self.buf_y[idx]])

    def reservoir_slots(self, n: int) -> list[tuple[int, int]]:
        """Reservoir sampling for the next n stream samples: (batch index, buffer slot) pairs
        for the samples that get stored."""
        out = []
        for i in range(n):
            if self.n_filled < self.mem_size:
                out.append((i, self.n_filled))
                self.n_filled += 1
            else:
                j = int(torch.randint(self.n_seen + 1, (1,), generator=self.gen))
                if j < self.mem_size:
                    out.append((i, j))
            self.n_seen += 1
        return out

    def after_step(self, x, y, task):
        # Only add samples on the first epoch, so each sample enters the stream once.
        if self.epoch > 0:
            return
        for i, j in self.reservoir_slots(len(y)):
            self.buf_x[j], self.buf_y[j] = x[i], y[i]

    def buffer_class_counts(self) -> list[int]:
        return torch.bincount(self.buf_y[:self.n_filled], minlength=N_CLASSES).tolist()


class DERPP(Replay):
    """DER++ (Buzzega et al., NeurIPS 2020, "Dark Experience for General Continual Learning").

    Same reservoir buffer as Replay, but each stored sample also keeps the logits the
    model produced for it when it was seen. Loss per step:
        CE(current batch)
      + der_alpha * MSE(model(buffer batch A), stored logits A)      # "dark knowledge"
      + der_beta  * CE(model(buffer batch B), labels B)              # plain replay term
    with A and B drawn independently from the buffer, each of batch_size samples.
    """
    name = "derpp"

    def __init__(self, *args, der_alpha: float = 0.5, der_beta: float = 0.5, **kw):
        super().__init__(*args, **kw)
        self.alpha, self.beta = der_alpha, der_beta
        self.buf_logits = torch.empty(self.mem_size, N_CLASSES, device=self.device)
        self._cur_logits: torch.Tensor | None = None

    def make_batch(self, x, y, task):
        return x, y  # replay enters through extra_loss, not by concatenation

    def task_loss(self, x, y, task):
        logits = self.model(x)
        self._cur_logits = logits.detach()  # stored with the sample (pre-update, as in the paper)
        return F.cross_entropy(self.masked_logits(logits, y), y)

    def extra_loss(self, x, y, task):
        if self.n_filled == 0:
            return 0.0
        a = self.sample_buffer(self.cfg.batch_size)
        loss = self.alpha * F.mse_loss(self.model(self.buf_x[a]), self.buf_logits[a])
        b = self.sample_buffer(self.cfg.batch_size)
        logits_b = self.masked_logits(self.model(self.buf_x[b]), self.buf_y[b])
        return loss + self.beta * F.cross_entropy(logits_b, self.buf_y[b])

    def after_step(self, x, y, task):
        if self.epoch > 0:
            return
        for i, j in self.reservoir_slots(len(y)):
            self.buf_x[j], self.buf_y[j], self.buf_logits[j] = x[i], y[i], self._cur_logits[i]


class EWC(Strategy):
    """Online EWC (Kirkpatrick et al. 2017; online variant of Schwarz et al. 2018).

    After each task, estimate the diagonal Fisher information F of the parameters on
    that task's data, add it to a running total, and anchor to the current weights θ*.
    While training later tasks, add  (λ/2) Σ F_i (θ_i - θ*_i)^2  to the loss, which
    makes weights that mattered for old tasks expensive to move.

    Fisher uses labels sampled from the model's own predictive distribution (the
    "true" Fisher, not the empirical one), with per-sample gradients via torch.func.
    """
    name = "ewc"

    def __init__(self, *args, ewc_lambda: float = 10000.0, fisher_samples: int = 1000,
                 ewc_gamma: float = 1.0, **kw):
        super().__init__(*args, **kw)
        self.lam = ewc_lambda
        self.fisher_samples = fisher_samples
        self.gamma = ewc_gamma  # decay of older Fisher terms; 1.0 = plain accumulation
        self.fisher: dict[str, torch.Tensor] | None = None
        self.anchor: dict[str, torch.Tensor] | None = None

    def extra_loss(self, x, y, task):
        if self.fisher is None:
            return 0.0
        pen = sum((self.stiffness[k] * (p - self.anchor[k]).pow(2)).sum()
                  for k, p in self.model.named_parameters())
        return 0.5 * pen

    def _update_stiffness(self) -> None:
        # Per-weight curvature of the penalty is λ·F. Gradient descent on a quadratic
        # diverges once lr·λ·F exceeds ~2, which happens for large λ (weights go NaN).
        # Cap it at 1/lr: at that point one step already pulls the weight straight back
        # to its anchor, so stiffer gives no extra protection, only instability.
        cap = 1.0 / self.cfg.lr
        self.stiffness = {k: (self.lam * f).clamp(max=cap) for k, f in self.fisher.items()}

    def after_task(self, task):
        n = min(self.fisher_samples, len(task.y_train))
        idx = torch.randperm(len(task.y_train), generator=self.gen)[:n]
        x = task.x_train[idx].to(self.device)
        y_task = task.y_train[idx].to(self.device)  # only used to pick the task-IL mask

        self.model.eval()
        params = {k: p.detach() for k, p in self.model.named_parameters()}
        with torch.no_grad():
            logits = self.masked_logits(self.model(x), y_task)
            y_sampled = torch.distributions.Categorical(logits=logits).sample()
        mask = self.same_task[y_task] if self.cfg.scenario == "task" else torch.ones_like(logits, dtype=torch.bool)

        def nll(p, xi, yi, mi):
            out = functional_call(self.model, p, (xi.unsqueeze(0),))
            out = out.masked_fill(~mi.unsqueeze(0), float("-inf"))
            return F.cross_entropy(out, yi.unsqueeze(0))

        fisher = {k: torch.zeros_like(v) for k, v in params.items()}
        # Per-sample gradients take chunk * n_params floats; keep that around 0.5 GB.
        n_params = sum(v.numel() for v in params.values())
        chunk = max(8, min(256, int(1.3e8 // n_params)))
        for i in range(0, n, chunk):
            g = vmap(grad(nll), in_dims=(None, 0, 0, 0))(params, x[i:i + chunk], y_sampled[i:i + chunk],
                                                         mask[i:i + chunk])
            for k in fisher:
                fisher[k] += g[k].pow(2).sum(0)
        self.model.train()

        if self.fisher is None:
            self.fisher = {k: f / n for k, f in fisher.items()}
        else:
            self.fisher = {k: self.gamma * self.fisher[k] + fisher[k] / n for k in fisher}
        self.anchor = {k: v.clone() for k, v in params.items()}
        self._update_stiffness()


STRATEGIES: dict[str, type[Strategy]] = {
    "naive": Naive,
    "replay": Replay,
    "ewc": EWC,
    "derpp": DERPP,
}

In [ ]:
%%writefile cl/ours.py
"""Our method: surprise-gated wake learning + sleep consolidation + judge with rollback.

Components (each can be switched off for ablations):

  use_surprise  Wake phase. Per-sample loss on each incoming batch is compared with
                an EMA estimate of the loss distribution (mean + surprise_z * std).
                Only samples above it ("surprising") give gradient and are stored in
                the fast buffer. A floor of `min_learn` highest-loss samples per batch
                keeps learning from stalling. Off: learn from / store every sample.
  (always)      Fast buffer: FIFO of the last `fast_size` stored (surprising) samples.
  (always)      Long-term memory (LTM): class-balanced reservoir. Every gated (stored)
                sample is reservoir-inserted into LTM the moment it is pushed to the fast
                buffer, so each first-epoch stream sample that passes the gate is an LTM
                storage candidate exactly once (as in Replay, but only gated samples).
  use_sleep     Every `sleep_every` wake steps and at task end: `sleep_steps` SGD steps
                on batches mixing new samples (fast buffer + current-task LTM) with
                old-task LTM samples. Off: no periodic sleep; wake steps instead append a
                plain replay batch (batch_size samples) drawn from old-task LTM.
  use_judge     After each update (each consolidation point) count correct predictions on
                the stored judge sets of OLD tasks. Criterion (integer counts, no float
                rounding): reject if ANY old task lost more than `judge_max_drop` correct
                judge samples versus the reference. Reference (`judge_ref`): "last_accept"
                = the last accepted checkpoint (ratchets), "task_start" = the state at the
                start of the current task (bounds drift within a task). Reject -> restore
                model + optimizer (deep copies) to the last accepted checkpoint and retry
                the update from memory. With use_sleep the retry is a sleep phase whose
                old-replay share is multiplied by `retry_boost` per retry (and, with
                use_targeted, degrading tasks get weight 1+retry_boost); without
                use_sleep the retry is plain wake-style replay (batch_size new from
                memory + batch_size old, no boost). After `max_retries` the retry result
                is accepted anyway so the new task is still learned. Off: never evaluate,
                never roll back, no judge storage.
  use_targeted  Old-task replay sampling weights w_t proportional to 1 + target_k * drop_t,
                drop_t = latest measured accuracy drop of task t vs its reference
                (judge sets if use_judge, else that task's LTM samples). Off: uniform
                everywhere, including retries.

Memory budget: fast buffer + LTM + stored judge sets <= mem_size at all times
(asserted after every change). LTM capacity is whatever is left, so turning the judge
off hands its share to LTM, keeping every ablation at the same 500-sample budget as
the replay baseline. Model / optimizer checkpoints are parameters, not samples.
"""
from __future__ import annotations

import copy
import math

import torch
import torch.nn.functional as F

from .data import TaskData
from .metrics import evaluate
from .strategies import STRATEGIES, Strategy

# Shared CLI hparams meant for other strategies; accepted and ignored.
_FOREIGN_HPARAMS = {"seed", "input_shape", "ewc_lambda", "fisher_samples", "ewc_gamma",
                    "der_alpha", "der_beta"}  # seed goes to the base class


class Ours(Strategy):
    name = "ours"

    def __init__(self, *args, mem_size: int = 500, fast_size: int = 50, judge_per_task: int = 40,
                 use_surprise: bool = True, use_sleep: bool = True, use_judge: bool = True,
                 use_targeted: bool = True,
                 surprise_z: float = -0.5, surprise_ema: float = 0.05, min_learn: int = 16,
                 sleep_every: int = 25, sleep_steps: int = 10, sleep_new: int = 64, sleep_old: int = 96,
                 wake_replay: int = 0,
                 judge_max_drop: int = 3, judge_threshold: float | None = None,
                 judge_ref: str = "task_start",
                 judge_criterion: str = "any", judge_until_acc: float | None = None,
                 judge_tradeoff: float = 0.0, rollback_mode: str = "full",
                 max_retries: int = 2, retry_boost: float = 2.0,
                 target_k: float = 5.0, **kw):
        unknown = set(kw) - _FOREIGN_HPARAMS
        if unknown:
            raise TypeError(f"Ours: unknown hparams {sorted(unknown)}")
        super().__init__(*args, **kw)
        self.mem_size, self.fast_size, self.judge_per_task = mem_size, fast_size, judge_per_task
        self.use_surprise, self.use_sleep = use_surprise, use_sleep
        self.use_judge, self.use_targeted = use_judge, use_targeted
        self.surprise_z, self.surprise_ema, self.min_learn = surprise_z, surprise_ema, min_learn
        self.sleep_every, self.sleep_steps = sleep_every, sleep_steps
        self.sleep_new, self.sleep_old, self.wake_replay = sleep_new, sleep_old, wake_replay
        # The judge compares integer counts. judge_threshold (a fraction), if given, overrides
        # judge_max_drop and is converted to samples with floor + epsilon, so e.g. 0.05 * 40 = 2
        # always means "reject a loss of 3+ samples", independent of float rounding.
        # Defaults (max_drop 3, task_start reference) were picked on val, seeds 0-2;
        # sleep_steps=10 (was 25) was picked on val, seeds 0-4 (same ACC, fewer steps, less forgetting).
        if judge_threshold is not None:
            judge_max_drop = int(math.floor(judge_threshold * judge_per_task + 1e-9))
        if judge_ref not in ("last_accept", "task_start"):
            raise ValueError(f"judge_ref must be last_accept or task_start, got {judge_ref!r}")
        self.judge_max_drop, self.judge_ref = int(judge_max_drop), judge_ref
        if judge_criterion not in ("any", "mean"):
            raise ValueError(f"judge_criterion must be any or mean, got {judge_criterion!r}")
        # Relaxed judge (defaults = original strict judge):
        #   judge_criterion "mean": reject only if the AVERAGE old-task loss exceeds judge_max_drop
        #   judge_until_acc X:      stop judging for the rest of a task once its accuracy reaches X
        #   judge_tradeoff λ > 0:   also accept if mean old-task accuracy lost since the last
        #                           checkpoint <= λ * new-task accuracy gained since it
        self.judge_criterion, self.judge_until_acc = judge_criterion, judge_until_acc
        self.judge_tradeoff = float(judge_tradeoff)
        # rollback_mode "full": a rejected update is undone back to the last accepted checkpoint,
        #   which also discards the wake (new-data) steps since then. On CIFAR-10 this threw away
        #   80% of all new-task training.
        # rollback_mode "sleep_only": checkpoint right after the wake steps, before sleep, so a
        #   rejection only undoes the sleep phase; the new-data learning is kept and the retries
        #   repair old tasks on top of it.
        if rollback_mode not in ("full", "sleep_only"):
            raise ValueError(f"rollback_mode must be full or sleep_only, got {rollback_mode!r}")
        self.rollback_mode = rollback_mode
        self.judge_off = False        # set when judge_until_acc is reached; reset per task
        self.ckpt_correct: dict[int, int] = {}
        self.ckpt_new_acc = 0.0
        self.max_retries, self.retry_boost = max_retries, retry_boost
        self.target_k = target_k
        if fast_size >= mem_size:
            raise ValueError("fast_size must leave room for long-term memory")

        d = self.device
        # fast buffer (FIFO ring)
        self.fx = torch.empty(fast_size, *self.input_shape, device=d)
        self.fy = torch.empty(fast_size, dtype=torch.long, device=d)
        self.f_moved = torch.zeros(fast_size, dtype=torch.bool, device=d)
        self.f_ptr = self.f_count = 0
        # long-term memory: per-class reservoirs {class: (x, y)}, plus stream counts
        self.ltm: dict[int, tuple[torch.Tensor, torch.Tensor]] = {}
        self.ltm_seen: dict[int, int] = {}
        self.class_task: dict[int, int] = {}
        self._flat = None  # cached (x, y, task) view of LTM
        # judge sets of finished tasks {task_id: (x, y)} and reference accuracies
        self.judge: dict[int, tuple[torch.Tensor, torch.Tensor]] = {}
        self.ref_correct: dict[int, int] = {}    # judge reference (counts), see judge_ref
        self.start_correct: dict[int, int] = {}  # judge counts at the start of the current task
        self.ltm_ref_acc: dict[int, float] = {}
        self.drop: dict[int, float] = {}
        # surprise statistics
        self.mu = self.var = None
        self.ckpt = None
        self.since_ckpt = {"wake": 0, "sleep": 0, "retry": 0}  # grad steps since last checkpoint
        self.stats = {"wake_steps": 0, "sleep_steps": 0, "retry_steps": 0, "grad_steps": 0,
                      "discarded_wake_steps": 0, "discarded_sleep_steps": 0, "discarded_retry_steps": 0,
                      "consolidations": 0, "judge_evals": 0, "rollbacks": 0, "retries": 0,
                      "forced_accepts": 0, "samples_seen": 0, "samples_learned": 0,
                      "samples_stored_fast": 0, "ltm_candidates": 0, "max_mem_used": 0,
                      "max_judge_drift": 0.0, "tradeoff_accepts": 0, "judge_stopped_tasks": 0,
                      "per_task": []}

    # ---- memory ----------------------------------------------------------
    def mem_used(self) -> int:
        return (self.f_count + sum(len(y) for _, y in self.ltm.values())
                + sum(len(y) for _, y in self.judge.values()))

    def _check_mem(self) -> None:
        used = self.mem_used()
        assert used <= self.mem_size, f"memory budget exceeded: {used} > {self.mem_size}"
        self.stats["max_mem_used"] = max(self.stats["max_mem_used"], used)

    def _ltm_cap_per_class(self) -> int:
        judge_n = sum(len(y) for _, y in self.judge.values())
        cap = self.mem_size - self.fast_size - judge_n
        return cap // max(1, len(self.ltm_seen))

    def _rebalance_ltm(self) -> None:
        cap = self._ltm_cap_per_class()
        for c, (x, y) in self.ltm.items():
            if len(y) > cap:  # reservoir contents are a random sample; keep a random subset
                keep = torch.randperm(len(y), generator=self.gen)[:cap].to(self.device)
                self.ltm[c] = (x[keep], y[keep])
        self._flat = None

    def _fast_push(self, x: torch.Tensor, y: torch.Tensor) -> None:
        """Store gated samples: FIFO fast buffer (recent pool for sleep) and, at the same
        time, a reservoir insert into LTM, so no stored sample is overwritten in the fast
        buffer before it has had its chance to enter LTM."""
        for i in range(len(y)):
            self.fx[self.f_ptr], self.fy[self.f_ptr] = x[i], y[i]
            self.f_moved[self.f_ptr] = False
            self._ltm_insert(self.f_ptr)
            self.f_ptr = (self.f_ptr + 1) % self.fast_size
            self.f_count = min(self.f_count + 1, self.fast_size)
        self.stats["samples_stored_fast"] += len(y)
        if len(y):
            self._flat = None
        self._check_mem()

    def _ltm_insert(self, i: int) -> None:
        """Reservoir-insert fast slot i into its class's LTM reservoir (once per sample)."""
        cap = self._ltm_cap_per_class()
        c = int(self.fy[i])
        x, y = self.ltm.get(c, (self.fx[:0], self.fy[:0]))
        self.ltm_seen[c] += 1
        self.stats["ltm_candidates"] += 1
        if len(y) < cap:
            self.ltm[c] = (torch.cat([x, self.fx[i:i + 1]]), torch.cat([y, self.fy[i:i + 1]]))
        else:
            j = int(torch.randint(self.ltm_seen[c], (1,), generator=self.gen))
            if j < cap:
                x[j], y[j] = self.fx[i], self.fy[i]
        self.f_moved[i] = True

    def _ltm_flat(self):
        if self._flat is None:
            if self.ltm:
                xs, ys = zip(*self.ltm.values())
                x, y = torch.cat(xs), torch.cat(ys)
            else:
                x, y = self.fx[:0], self.fy[:0]
            t = torch.tensor([self.class_task[int(c)] for c in y.tolist()], dtype=torch.long,
                             device=self.device)
            self._flat = (x, y, t)
        return self._flat

    def _rand(self, n: int, k: int) -> torch.Tensor:
        return torch.randint(n, (k,), generator=self.gen).to(self.device)

    def _sample_old(self, k: int, cur: int, boost: dict[int, float] | None = None):
        """k samples from LTM of tasks < cur, task chosen by (targeted) weights, uniform inside."""
        x, y, t = self._ltm_flat()
        old = sorted({int(v) for v in t.tolist() if v < cur})
        if k <= 0 or not old:
            return None
        w = torch.ones(len(old))
        if self.use_targeted:
            w = torch.tensor([1.0 + self.target_k * max(0.0, self.drop.get(o, 0.0)) for o in old])
        if boost:
            w = w * torch.tensor([boost.get(o, 1.0) for o in old])
        pick = torch.multinomial(w / w.sum(), k, replacement=True, generator=self.gen)
        out = torch.empty(k, dtype=torch.long)
        for a, o in enumerate(old):
            sel = (pick == a).nonzero().flatten()
            if len(sel):
                pool = (t == o).nonzero().flatten().cpu()
                out[sel] = pool[torch.randint(len(pool), (len(sel),), generator=self.gen)]
        out = out.to(self.device)
        return x[out], y[out]

    def _sample_new(self, k: int, cur: int):
        """k samples from the fast buffer plus the current task's share of LTM."""
        x, y, t = self._ltm_flat()
        m = t == cur
        px = torch.cat([self.fx[:self.f_count], x[m]])
        py = torch.cat([self.fy[:self.f_count], y[m]])
        if k <= 0 or len(py) == 0:
            return None
        i = self._rand(len(py), k)
        return px[i], py[i]

    # ---- training pieces ---------------------------------------------------
    def _step(self, loss: torch.Tensor, kind: str) -> None:
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.optimizer.step()
        self.stats[f"{kind}_steps"] += 1
        self.stats["grad_steps"] += 1
        self.since_ckpt[kind] += 1

    def _gate(self, losses: torch.Tensor) -> torch.Tensor:
        """Boolean mask of samples to learn from / store."""
        l = losses.detach()
        if not self.use_surprise:
            return torch.ones_like(l, dtype=torch.bool)
        if self.mu is None:
            self.mu, self.var = l.mean().item(), l.var().item() if len(l) > 1 else 0.0
        thr = self.mu + self.surprise_z * self.var ** 0.5
        sel = l > thr
        k = min(self.min_learn, len(l))
        if int(sel.sum()) < k:
            sel = torch.zeros_like(sel)
            sel[l.topk(k).indices] = True
        a = self.surprise_ema
        self.mu = (1 - a) * self.mu + a * l.mean().item()
        self.var = (1 - a) * self.var + a * (l.var().item() if len(l) > 1 else 0.0)
        return sel

    def _wake_step(self, x, y, task: TaskData) -> None:
        losses = F.cross_entropy(self.masked_logits(self.model(x), y), y, reduction="none")
        sel = self._gate(losses)
        total, n = losses[sel].sum(), int(sel.sum())
        n_rep = self.wake_replay if self.use_sleep else len(y)
        rep = self._sample_old(n_rep, task.task_id)
        if rep is not None:
            rx, ry = rep
            total = total + F.cross_entropy(self.masked_logits(self.model(rx), ry), ry, reduction="sum")
            n += len(ry)
        self._step(total / n, "wake")
        self.stats["samples_seen"] += len(y)
        self.stats["samples_learned"] += int(sel.sum())
        if self.epoch == 0:  # each stream sample is a storage candidate once, as in Replay
            self._fast_push(x[sel], y[sel])

    def _sleep(self, task: TaskData, steps: int, old_mult: float = 1.0,
               boost: dict[int, float] | None = None, kind: str = "sleep",
               n_new: int | None = None, n_old: int | None = None) -> None:
        n_new = self.sleep_new if n_new is None else n_new
        n_old = self.sleep_old if n_old is None else n_old
        for _ in range(steps):
            parts = [self._sample_new(n_new, task.task_id),
                     self._sample_old(int(round(n_old * old_mult)), task.task_id, boost)]
            parts = [p for p in parts if p is not None]
            if not parts:
                return
            x = torch.cat([p[0] for p in parts])
            y = torch.cat([p[1] for p in parts])
            self._step(F.cross_entropy(self.masked_logits(self.model(x), y), y), kind)

    # ---- judge -------------------------------------------------------------
    def _judge_correct(self) -> dict[int, int]:
        """Number of correctly classified judge samples per old task (integer)."""
        return {t: int(round(evaluate(self.model, x, y, self.device, classes=self.seen_tasks[t].classes,
                                      scenario=self.cfg.scenario) * len(y)))
                for t, (x, y) in self.judge.items()}

    def _ltm_accs(self, cur: int) -> dict[int, float]:
        x, y, t = self._ltm_flat()
        out = {}
        for o in sorted({int(v) for v in t.tolist() if v < cur}):
            m = t == o
            out[o] = evaluate(self.model, x[m], y[m], self.device, classes=self.seen_tasks[o].classes,
                              scenario=self.cfg.scenario)
        return out

    def _new_acc(self, task: TaskData) -> float:
        """Accuracy on the current task's samples held in memory (fast buffer + its LTM share).
        These are training samples, so it is an optimistic estimate; it is only used as a
        relative signal (did the update help the new task?)."""
        x, y, t = self._ltm_flat()
        m = t == task.task_id
        px = torch.cat([self.fx[:self.f_count], x[m]])
        py = torch.cat([self.fy[:self.f_count], y[m]])
        if len(py) == 0:
            return 0.0
        return evaluate(self.model, px, py, self.device, classes=task.classes, scenario=self.cfg.scenario)

    def _relaxed(self) -> bool:
        return self.judge_until_acc is not None or self.judge_tradeoff > 0

    def _save_ckpt(self) -> None:
        self.ckpt = (copy.deepcopy(self.model.state_dict()), copy.deepcopy(self.optimizer.state_dict()))
        self.since_ckpt = dict.fromkeys(self.since_ckpt, 0)

    def _restore_ckpt(self) -> None:
        # model.load_state_dict copies into the parameters, but Optimizer.load_state_dict puts
        # the checkpoint's own momentum tensors into the live optimizer, which then mutates them
        # in place. Load a deep copy so the checkpoint stays clean across repeated rollbacks.
        self.model.load_state_dict(self.ckpt[0])
        self.optimizer.load_state_dict(copy.deepcopy(self.ckpt[1]))
        for k, v in self.since_ckpt.items():
            self.stats[f"discarded_{k}_steps"] += v
        self.since_ckpt = dict.fromkeys(self.since_ckpt, 0)

    def _consolidate(self, task: TaskData) -> None:
        """One consolidation point: transfer, sleep, judge (with rollback / retry)."""
        if self.rollback_mode == "sleep_only" and self.use_judge and self.judge and not self.judge_off:
            self._save_ckpt()  # keep the wake steps: a rollback only undoes what follows
        if self.use_sleep:
            self._sleep(task, self.sleep_steps)
        self.stats["consolidations"] += 1
        pt = self.stats["per_task"][-1]

        if not self.use_judge:
            if self.use_targeted:
                accs = self._ltm_accs(task.task_id)
                self.drop = {t: self.ltm_ref_acc.get(t, a) - a for t, a in accs.items()}
            return
        if not self.judge:
            self._save_ckpt()
            return
        n_j = {t: len(y) for t, (_, y) in self.judge.items()}
        new_acc = self._new_acc(task) if self._relaxed() else 0.0
        if self.judge_off or (self.judge_until_acc is not None and new_acc >= self.judge_until_acc):
            # new lesson is learned well enough: stop judging (no rollback) for the rest of it
            if not self.judge_off:
                self.judge_off = True
                self.stats["judge_stopped_tasks"] += 1
            correct = self._judge_correct()  # still measured, for targeted replay weights
            self.drop = {t: (self.ref_correct[t] - c) / n_j[t] for t, c in correct.items()}
            return
        for attempt in range(self.max_retries + 1):
            correct = self._judge_correct()
            self.stats["judge_evals"] += 1
            lost = {t: self.ref_correct[t] - c for t, c in correct.items()}
            self.drop = {t: d / n_j[t] for t, d in lost.items()}
            worst = max(lost.values()) if self.judge_criterion == "any" else sum(lost.values()) / len(lost)
            if worst <= self.judge_max_drop:
                break
            if self.judge_tradeoff > 0:
                new_acc = self._new_acc(task)
                old_cost = sum((self.ckpt_correct.get(t, c) - c) / n_j[t] for t, c in correct.items()) / len(correct)
                if old_cost <= self.judge_tradeoff * (new_acc - self.ckpt_new_acc):
                    self.stats["tradeoff_accepts"] += 1
                    break
            if attempt == self.max_retries:
                self.stats["forced_accepts"] += 1
                pt["forced_accepts"] += 1
                break
            # reject: roll back and retry from memory with more replay of degrading tasks
            self._restore_ckpt()
            self.stats["rollbacks"] += 1
            self.stats["retries"] += 1
            pt["rollbacks"] += 1
            if self.use_sleep:
                boost = None
                if self.use_targeted:
                    boost = {t: (1.0 + self.retry_boost) if d > self.judge_max_drop else 1.0
                             for t, d in lost.items()}
                self._sleep(task, self.sleep_steps, old_mult=self.retry_boost ** (attempt + 1),
                            boost=boost, kind="retry")
            else:  # no sleep: retry with plain wake-style replay from memory, no boost
                bs = self.cfg.batch_size
                self._sleep(task, self.sleep_steps, n_new=bs, n_old=bs, kind="retry")
        # accept: new checkpoint; with judge_ref="last_accept" the reference ratchets
        if self.judge_ref == "last_accept":
            self.ref_correct.update(correct)
        drift = max((self.start_correct[t] - c) / n_j[t] for t, c in correct.items())
        pt["max_judge_drift"] = max(pt["max_judge_drift"], drift)
        self.stats["max_judge_drift"] = max(self.stats["max_judge_drift"], drift)
        self.ckpt_correct = dict(correct)
        if self.judge_tradeoff > 0:
            self.ckpt_new_acc = self._new_acc(task)
        self._save_ckpt()

    # ---- task loop ---------------------------------------------------------
    def before_task(self, task: TaskData) -> None:
        super().before_task(task)
        for c in task.classes:
            self.class_task[c] = task.task_id
            self.ltm_seen.setdefault(c, 0)
        self._rebalance_ltm()
        self._check_mem()
        self.stats["per_task"].append({"task": task.task_id, "rollbacks": 0, "forced_accepts": 0,
                                       "max_judge_drift": 0.0})
        self.judge_off = False
        self.ckpt_correct = dict(self.start_correct)
        self.ckpt_new_acc = 0.0
        if self.use_judge:
            self._save_ckpt()  # fresh optimizer -> checkpoint must hold its (empty) state

    def after_task(self, task: TaskData) -> None:
        self._consolidate(task)
        self.f_count = self.f_ptr = 0  # everything is in LTM now; fast buffer is per-stream
        self.f_moved.zero_()
        if self.use_judge and self.judge_per_task > 0:
            if task.y_judge is None or len(task.y_judge) < self.judge_per_task:
                raise ValueError("task has fewer judge samples than judge_per_task")
            self.judge[task.task_id] = (task.x_judge[:self.judge_per_task].to(self.device),
                                        task.y_judge[:self.judge_per_task].to(self.device))
            self._rebalance_ltm()  # make room before the budget check
        self._check_mem()

    def train_task(self, task: TaskData) -> None:
        self.before_task(task)
        self.model.train()
        pt = self.stats["per_task"][-1]
        g0 = self.stats["grad_steps"]
        # the task becomes visible in seen_tasks only at the end, but judge/LTM helpers
        # index seen_tasks by task id, which is fine because they only touch old tasks.
        n, wake = len(task.y_train), 0
        for self.epoch in range(self.cfg.epochs):
            perm = torch.randperm(n, generator=self.gen)
            for i in range(0, n, self.cfg.batch_size):
                idx = perm[i:i + self.cfg.batch_size]
                self._wake_step(task.x_train[idx].to(self.device), task.y_train[idx].to(self.device), task)
                wake += 1
                if wake % self.sleep_every == 0:
                    self._consolidate(task)
        self.after_task(task)
        self.seen_tasks.append(task)
        if self.use_judge:  # references for the next task: counts at the accepted end state
            self.start_correct = self._judge_correct()  # (checkpoint is taken in before_task)
            self.ref_correct = dict(self.start_correct)
        self.ltm_ref_acc = self._ltm_accs(task.task_id + 1)
        self.drop = {}
        pt["grad_steps"] = self.stats["grad_steps"] - g0
        pt["mem_used_end"] = self.mem_used()


STRATEGIES["ours"] = Ours

In [ ]:
%%writefile run_experiment.py
"""Run a continual-learning strategy on Split MNIST / Split CIFAR-10 and record the accuracy matrix.

Example:
    .venv/bin/python run_experiment.py --strategy naive --seeds 0 1 2
    .venv/bin/python run_experiment.py --dataset cifar10 --strategy derpp --seeds 0

For each seed: train on tasks 0..4 in order; after each task, evaluate test
accuracy on every task seen so far. Prints the accuracy matrix and writes
results/<strategy>_<scenario>.json with per-seed matrices and summary metrics.
"""
from __future__ import annotations

import argparse
import ast
import json
import random
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from cl.data import DATASETS, make_split
from cl.metrics import (SCENARIOS, avg_acc_curve, evaluate, final_avg_acc, forgetting,
                        format_matrix, new_acc_matrix)
from cl.model import MLP, ReducedResNet18
from cl.strategies import STRATEGIES, TrainConfig
import cl.ours  # noqa: F401,E402  (registers "ours" in STRATEGIES)

HERE = Path(__file__).resolve().parent


def pick_device(name: str, dataset: str = "mnist") -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    # The MNIST MLP is faster on CPU than on Apple MPS; the CIFAR ResNet is ~7x faster on MPS.
    if dataset == "cifar10" and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def make_model(dataset: str) -> torch.nn.Module:
    return MLP() if dataset == "mnist" else ReducedResNet18()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_one(strategy_name: str, cfg: TrainConfig, seed: int, device: torch.device,
            n_tasks: int, hparams: dict | None = None, eval_split: str = "test",
            verbose: bool = True, return_stats: bool = False, dataset: str = "mnist"):
    set_seed(seed)
    tasks = make_split(dataset, root=str(HERE / "data"), n_tasks=n_tasks, seed=seed)
    strategy = STRATEGIES[strategy_name](make_model(dataset), cfg, device, seed=seed,
                                         input_shape=tuple(tasks[0].x_train.shape[1:]),
                                         **(hparams or {}))
    R = new_acc_matrix(n_tasks)

    for i, task in enumerate(tasks):
        t0 = time.time()
        strategy.train_task(task)
        for j in range(i + 1):
            x, y = getattr(tasks[j], f"x_{eval_split}"), getattr(tasks[j], f"y_{eval_split}")
            R[i, j] = evaluate(strategy.model, x, y, device,
                               classes=tasks[j].classes, scenario=cfg.scenario)
        if verbose:
            accs = " ".join(f"T{j}={R[i, j] * 100:5.1f}" for j in range(i + 1))
            print(f"  [seed {seed}] after task {i} {task.classes} ({time.time() - t0:4.1f}s): {accs}")
    if return_stats:
        return R, getattr(strategy, "stats", None)
    return R


def parse_hp(items: list[str]) -> dict:
    """--hp key=value pairs; values via ast.literal_eval, falling back to the raw string."""
    out = {}
    for item in items:
        if "=" not in item:
            raise SystemExit(f"--hp expects key=value, got {item!r}")
        k, v = item.split("=", 1)
        try:
            out[k.strip()] = ast.literal_eval(v)
        except (ValueError, SyntaxError):
            out[k.strip()] = v
    return out


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--dataset", default="mnist", choices=DATASETS)
    p.add_argument("--strategy", default="naive", choices=sorted(STRATEGIES))
    p.add_argument("--scenario", default="class", choices=SCENARIOS)
    p.add_argument("--seeds", type=int, nargs="+", default=[0])
    p.add_argument("--n-tasks", type=int, default=5)
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--lr", type=float, default=0.01)
    p.add_argument("--device", default="auto", help="auto | cpu | cuda | mps")
    p.add_argument("--eval-split", default="test", choices=("test", "val"),
                   help="use 'val' for hyperparameter tuning so the test set stays untouched")
    p.add_argument("--mem-size", type=int, default=500, help="replay: total buffer size")
    p.add_argument("--ewc-lambda", type=float, default=10000.0, help="ewc: penalty strength")
    p.add_argument("--fisher-samples", type=int, default=1000, help="ewc: samples per Fisher estimate")
    p.add_argument("--der-alpha", type=float, default=0.5, help="derpp: weight of the logit-matching term")
    p.add_argument("--der-beta", type=float, default=0.5, help="derpp: weight of the replay CE term")
    p.add_argument("--hp", action="append", default=[], metavar="KEY=VALUE",
                   help="extra strategy hparam (repeatable; value parsed with ast.literal_eval), "
                        "e.g. --hp use_judge=False --hp sleep_steps=40")
    p.add_argument("--tag", default="", help="suffix for the output filename")
    p.add_argument("--quiet", action="store_true", help="only print the summary line")
    p.add_argument("--out", default=None, help="default: results/ (mnist) or results/cifar10/")
    args = p.parse_args()

    cfg = TrainConfig(epochs=args.epochs, batch_size=args.batch_size, lr=args.lr, scenario=args.scenario)
    hparams = {"mem_size": args.mem_size, "ewc_lambda": args.ewc_lambda,
               "fisher_samples": args.fisher_samples, "der_alpha": args.der_alpha, "der_beta": args.der_beta}
    hparams.update(parse_hp(args.hp))
    device = pick_device(args.device, args.dataset)
    print(f"dataset={args.dataset} strategy={args.strategy} scenario={args.scenario} eval={args.eval_split} device={device} "
          f"cfg={asdict(cfg)} hparams={hparams}")

    runs = []
    for seed in args.seeds:
        t0 = time.time()
        R, stats = run_one(args.strategy, cfg, seed, device, args.n_tasks, hparams, args.eval_split,
                           verbose=not args.quiet, return_stats=True, dataset=args.dataset)
        run = {"seed": seed, "acc_matrix": R.tolist(), "avg_acc_curve": avg_acc_curve(R).tolist(),
               "final_avg_acc": final_avg_acc(R), "forgetting": forgetting(R),
               "runtime_s": time.time() - t0}
        if stats is not None:
            run["stats"] = stats
            if not args.quiet:
                print("  stats: " + json.dumps({k: v for k, v in stats.items() if k != "per_task"}))
        runs.append(run)
        if not args.quiet:
            print(format_matrix(R))
        print(f"  final avg acc = {final_avg_acc(R) * 100:.2f}%   forgetting = {forgetting(R) * 100:.2f}%\n")

    accs = np.array([r["final_avg_acc"] for r in runs])
    fgts = np.array([r["forgetting"] for r in runs])
    summary = {"final_avg_acc_mean": accs.mean(), "final_avg_acc_std": accs.std(),
               "forgetting_mean": fgts.mean(), "forgetting_std": fgts.std()}
    print(f"== {args.strategy} ({args.scenario}-IL, {len(runs)} seed(s)): "
          f"ACC {accs.mean() * 100:.2f} ± {accs.std() * 100:.2f}%  "
          f"forgetting {fgts.mean() * 100:.2f} ± {fgts.std() * 100:.2f}%  [{args.eval_split}]")

    out = Path(args.out) if args.out else (HERE / "results" if args.dataset == "mnist"
                                           else HERE / "results" / args.dataset)
    out.mkdir(parents=True, exist_ok=True)
    suffix = (f"_{args.tag}" if args.tag else "") + ("_val" if args.eval_split == "val" else "")
    path = out / f"{args.strategy}_{args.scenario}{suffix}.json"
    path.write_text(json.dumps({"dataset": args.dataset, "strategy": args.strategy, "config": asdict(cfg), "hparams": hparams,
                                "eval_split": args.eval_split, "device": str(device),
                                "n_tasks": args.n_tasks, "summary": summary, "runs": runs}, indent=2))
    print(f"saved {path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run_suite.py
"""Benchmark suite v2: DER++, compute-matched baselines, and Split CIFAR-10.

Runs two chains at the same time (MNIST on CPU, CIFAR-10 on the GPU/MPS) and logs
progress to results/suite_progress.log. Every hyperparameter choice is made on the
val split; test runs only happen after the choice is fixed.

    .venv/bin/python run_suite.py            # everything
    .venv/bin/python run_suite.py --only mnist
    .venv/bin/python run_suite.py --only relax_mnist        # round 3: relaxed judge
    .venv/bin/python run_suite.py --only relax_cifar10      # (on a GPU, e.g. Kaggle)
    .venv/bin/python run_suite.py --only round4_cifar10     # round 4: keep-wake rollback (GPU)
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
import threading
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
PY = sys.executable
LOG = HERE / "results" / "suite_progress.log"
SWEEP = HERE / "results" / "sweep_v2"
_lock = threading.Lock()


def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    with _lock:
        print(line, flush=True)
        with LOG.open("a") as f:
            f.write(line + "\n")


def run(args: list[str], name: str) -> dict | None:
    """Run run_experiment.py; return the saved JSON (or None on failure)."""
    t0 = time.time()
    p = subprocess.run([PY, str(HERE / "run_experiment.py"), "--quiet", *args],
                       cwd=HERE, capture_output=True, text=True)
    saved = [l.split("saved ", 1)[1] for l in p.stdout.splitlines() if l.startswith("saved ")]
    if p.returncode or not saved:
        log(f"FAILED {name} ({time.time() - t0:.0f}s): {(p.stderr or p.stdout)[-600:]}")
        return None
    res = json.loads(Path(saved[0]).read_text())
    s = res["summary"]
    log(f"{name}: ACC {s['final_avg_acc_mean'] * 100:.2f} ± {s['final_avg_acc_std'] * 100:.2f}  "
        f"forgetting {s['forgetting_mean'] * 100:.2f}  ({time.time() - t0:.0f}s)")
    return res


def acc(res: dict | None) -> float:
    return -1.0 if res is None else res["summary"]["final_avg_acc_mean"]


def steps(res: dict | None) -> float:
    runs = (res or {}).get("runs", [])
    vals = [r["stats"]["grad_steps"] for r in runs if "stats" in r]
    return sum(vals) / len(vals) if vals else float("nan")


# ---------------------------------------------------------------- MNIST (CPU)
def mnist_chain() -> None:
    base = ["--dataset", "mnist", "--device", "cpu"]
    log("MNIST: tuning DER++ alpha/beta on val (seeds 0 1)")
    best, best_cfg = -1.0, None
    for a in (0.1, 0.2, 0.5, 1.0):
        for b in (0.5, 1.0):
            r = run(base + ["--strategy", "derpp", "--der-alpha", str(a), "--der-beta", str(b),
                            "--eval-split", "val", "--seeds", "0", "1", "--out", str(SWEEP / "mnist"),
                            "--tag", f"a{a}_b{b}"], f"mnist derpp val a={a} b={b}")
            if acc(r) > best:
                best, best_cfg = acc(r), (a, b)
    a, b = best_cfg
    log(f"MNIST: DER++ chosen alpha={a} beta={b} (val ACC {best * 100:.2f})")
    der = ["--der-alpha", str(a), "--der-beta", str(b)]

    for sc in ("class", "task"):
        run(base + ["--strategy", "derpp", "--scenario", sc, "--seeds", "0", "1", "2"] + der,
            f"mnist derpp {sc} test")
    # Compute-matched: ours uses ~2.8k grad steps vs 1.7k at 2 epochs; 4 epochs = 3.4k (>= ours).
    for st in ("replay", "derpp"):
        run(base + ["--strategy", st, "--epochs", "4", "--seeds", "0", "1", "2", "--tag", "epochs4"] + der,
            f"mnist {st} class test, 4 epochs (compute-matched)")
    log("MNIST chain done")


# ------------------------------------------------------------ CIFAR-10 (MPS)
def cifar_chain() -> None:
    E = "5"  # epochs per task (the literature uses 50; too slow for a laptop)
    base = ["--dataset", "cifar10", "--epochs", E]
    log("CIFAR: choosing learning rate with replay on val (seed 0)")
    best, lr = -1.0, None
    for cand in ("0.01", "0.03", "0.1"):
        r = run(base + ["--strategy", "replay", "--lr", cand, "--eval-split", "val", "--seeds", "0",
                        "--out", str(SWEEP / "cifar10"), "--tag", f"lr{cand}"], f"cifar replay val lr={cand}")
        if acc(r) > best:
            best, lr = acc(r), cand
    log(f"CIFAR: lr={lr} (val ACC {best * 100:.2f})")
    base += ["--lr", lr]

    log("CIFAR: choosing DER++ alpha on val (seed 0)")
    best, alpha = -1.0, None
    for a in ("0.1", "0.3", "1.0"):
        r = run(base + ["--strategy", "derpp", "--der-alpha", a, "--der-beta", "0.5", "--eval-split", "val",
                        "--seeds", "0", "--out", str(SWEEP / "cifar10"), "--tag", f"a{a}"],
                f"cifar derpp val alpha={a}")
        if acc(r) > best:
            best, alpha = acc(r), a
    log(f"CIFAR: DER++ alpha={alpha} (val ACC {best * 100:.2f})")
    der = ["--der-alpha", alpha, "--der-beta", "0.5"]

    seeds = ["--seeds", "0", "1", "2"]
    res = {}
    for st in ("naive", "replay", "derpp", "ours", "ewc"):
        res[st] = run(base + ["--strategy", st] + seeds + der, f"cifar {st} class test")
    # Only the judge ablation on CIFAR (the judge is the least-published component); the
    # full ablation set is on MNIST. Each CIFAR "ours" run is ~9 min per seed on MPS.
    for tag, hp in (("no-judge", "use_judge=False"),):
        run(base + ["--strategy", "ours", "--hp", hp, "--tag", tag] + seeds, f"cifar ours {tag} class test")

    # Compute-matched baselines: scale epochs so their grad steps >= ours.
    ratio = steps(res["ours"]) / steps(res["replay"]) if res.get("ours") and res.get("replay") else 2.0
    e2 = str(max(int(E) + 1, int(-(-int(E) * ratio // 1))))
    log(f"CIFAR: ours/replay grad-step ratio {ratio:.2f} -> compute-matched epochs {e2}")
    for st in ("replay", "derpp"):
        run(["--dataset", "cifar10", "--epochs", e2, "--lr", lr, "--strategy", st, "--tag", f"epochs{e2}"]
            + seeds + der, f"cifar {st} class test, {e2} epochs (compute-matched)")
    log("CIFAR chain done")


# ------------------------------------------- relaxed judge (round 3)
# The strict judge keeps new lessons under-learned on CIFAR (diagonal 50-68% vs 80-95% for
# the baselines). These configs relax it in three ways; "strict" is the original default.
RELAX_CONFIGS = {
    "mnist": {
        "strict": [],
        **{f"any_d{d}": ["--hp", f"judge_max_drop={d}"] for d in (5, 8)},
        **{f"mean_d{d}": ["--hp", "judge_criterion='mean'", "--hp", f"judge_max_drop={d}"] for d in (3, 5, 8)},
        **{f"until{a}": ["--hp", f"judge_until_acc={a}"] for a in (0.8, 0.9, 0.95, 0.98)},
        **{f"trade{l}": ["--hp", f"judge_tradeoff={l}"] for l in (0.25, 0.5, 1.0, 2.0)},
        "mean_d3_trade0.5": ["--hp", "judge_criterion='mean'", "--hp", "judge_tradeoff=0.5"],
    },
    # CIFAR runs are ~6.5 min per seed on a T4, so one representative per family.
    "cifar10": {
        "strict": [],
        "any_d5": ["--hp", "judge_max_drop=5"],
        "any_d8": ["--hp", "judge_max_drop=8"],
        "mean_d5": ["--hp", "judge_criterion='mean'", "--hp", "judge_max_drop=5"],
        "trade0.5": ["--hp", "judge_tradeoff=0.5"],
        "until0.8": ["--hp", "judge_until_acc=0.8"],
    },
}


def relax_chain(dataset: str) -> None:
    if dataset == "mnist":
        base, val_seeds = ["--dataset", "mnist", "--device", "cpu"], ["0", "1", "2"]
    else:  # same lr / epochs as the round-2 CIFAR runs
        base, val_seeds = ["--dataset", "cifar10", "--epochs", "5", "--lr", "0.03"], ["0", "1"]
    log(f"{dataset}: tuning the relaxed judge on val (seeds {' '.join(val_seeds)})")
    best, best_tag = -1.0, None
    for tag, hp in RELAX_CONFIGS[dataset].items():
        r = run(base + ["--strategy", "ours", "--eval-split", "val", "--seeds", *val_seeds,
                        "--out", str(HERE / "results" / "sweep_relax" / dataset), "--tag", tag] + hp,
                f"{dataset} ours val {tag}")
        if acc(r) > best:
            best, best_tag = acc(r), tag
    log(f"{dataset}: relaxed judge chosen = {best_tag} (val ACC {best * 100:.2f})")
    if best_tag == "strict":
        log(f"{dataset}: the strict judge won on val; no new test run needed")
        return
    run(base + ["--strategy", "ours", "--seeds", "0", "1", "2", "--tag", f"relaxed-{best_tag}"]
        + RELAX_CONFIGS[dataset][best_tag], f"{dataset} ours class test, relaxed judge ({best_tag})")
    log(f"{dataset} relax chain done")


# ------------------------------------------- round 4: keep new learning on rollback
# Round 3 showed rollbacks discard 80% of the new-data (wake) training on CIFAR-10, so new
# tasks stall near 65%. rollback_mode="sleep_only" keeps the wake steps and only redoes the
# sleep phase. Also tests the surprise gate (the other suspect). Variants are chosen on val
# (seeds 0 1); the strict default scored 27.75 on the same val split in round 3
# (results/sweep_relax/cifar10/ours_class_strict_val.json), so a variant is only run on test
# if it beats that.
ROUND4_CONFIGS = {
    "no-surprise": ["--hp", "use_surprise=False"],
    "keep-wake": ["--hp", "rollback_mode='sleep_only'"],
    "keep-wake-wr64": ["--hp", "rollback_mode='sleep_only'", "--hp", "wake_replay=64"],
}
ROUND4_STRICT_VAL = 0.2775


def round4_chain() -> None:
    base = ["--dataset", "cifar10", "--epochs", "5", "--lr", "0.03", "--strategy", "ours"]
    log("cifar10 round 4: keep-wake rollback and surprise gate, on val (seeds 0 1)")
    best, best_tag = -1.0, None
    for tag, hp in ROUND4_CONFIGS.items():
        r = run(base + ["--eval-split", "val", "--seeds", "0", "1",
                        "--out", str(HERE / "results" / "sweep_round4" / "cifar10"), "--tag", tag] + hp,
                f"cifar10 ours val {tag}")
        if acc(r) > best:
            best, best_tag = acc(r), tag
    log(f"cifar10 round 4: best variant = {best_tag} (val ACC {best * 100:.2f}; strict was "
        f"{ROUND4_STRICT_VAL * 100:.2f})")
    if best <= ROUND4_STRICT_VAL:
        log("cifar10 round 4: no variant beat the strict judge on val; no test run")
    else:
        run(base + ["--seeds", "0", "1", "2", "--tag", best_tag] + ROUND4_CONFIGS[best_tag],
            f"cifar10 ours class test, {best_tag}")
    log("cifar10 round 4 chain done")


CHAINS = {
    "mnist": mnist_chain,
    "cifar10": cifar_chain,
    "relax_mnist": lambda: relax_chain("mnist"),
    "relax_cifar10": lambda: relax_chain("cifar10"),
    "round4_cifar10": round4_chain,
}


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--only", nargs="+", choices=sorted(CHAINS),
                   help="chains to run (default: mnist and cifar10, i.e. round 2)")
    args = p.parse_args()
    LOG.parent.mkdir(parents=True, exist_ok=True)
    only = args.only or ["mnist", "cifar10"]
    log(f"suite start (chains={' '.join(only)})")
    chains = [CHAINS[n] for n in only]
    threads = [threading.Thread(target=c) for c in chains]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    log("suite done")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_report.py
"""Build the results report from results/*.json.

    .venv/bin/python make_report.py [--results results]

Reads the top-level JSON files in the results folder (subfolders and files with
eval_split == "val" are skipped) and writes:

  results/summary.md      one markdown table per scenario (also printed to stdout)
  results/avg_acc.png     average accuracy over seen tasks after each task, per scenario
  results/forgetting.png  final accuracy and forgetting per main method, per scenario

Each file is labelled by its filename stem (e.g. replay_class, ours_class_no-judge).
A file is a "main" method when its stem is exactly <strategy>_<scenario>; anything
after that is a tag and marks the file as a variant/ablation. Variants appear in
the table only, after the main methods.
"""
from __future__ import annotations

import argparse
import json
import math
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib.ticker import PercentFormatter  # noqa: E402

HERE = Path(__file__).resolve().parent
SCENARIO_ORDER = ("class", "task")
SCENARIO_TITLE = {"class": "Class-incremental", "task": "Task-incremental"}
MAIN_ORDER = ("naive", "ewc", "replay", "derpp", "ours")
METHOD_NAME = {"naive": "Naive fine-tuning", "ewc": "Online EWC", "replay": "Replay", "derpp": "DER++", "ours": "Ours"}
# Okabe-Ito palette (colour-blind safe); fixed per method so both figures agree.
METHOD_COLOR = {"naive": "#999999", "ewc": "#56B4E9", "replay": "#E69F00", "derpp": "#009E73", "ours": "#D55E00"}
EXTRA_COLORS = ["#CC79A7", "#0072B2", "#F0E442", "#000000"]
# Optional stats columns: key in the JSON -> column header.
STAT_COLUMNS = {"grad_steps": "grad steps", "rollbacks": "rollbacks"}


@dataclass
class Result:
    label: str          # filename stem
    strategy: str
    scenario: str
    tag: str            # "" for main methods
    runs: list
    stats: dict = field(default_factory=dict)  # stat key -> list of per-seed values (or one value)

    @property
    def is_main(self) -> bool:
        return self.tag == ""

    def per_seed(self, key: str) -> np.ndarray:
        return np.array([r[key] for r in self.runs], dtype=float)

    @property
    def curves(self) -> np.ndarray:
        return np.array([r["avg_acc_curve"] for r in self.runs], dtype=float)


def _stat_values(raw) -> list[float]:
    """Accept a scalar, a list of per-seed values, or {'mean': .., 'std': ..}."""
    if isinstance(raw, dict):
        if "values" in raw:
            return [float(v) for v in raw["values"]]
        if "mean" in raw:
            return [float(raw["mean"])]
        return []
    if isinstance(raw, (list, tuple)):
        return [float(v) for v in raw if isinstance(v, (int, float))]
    if isinstance(raw, (int, float)):
        return [float(raw)]
    return []


def collect_stats(d: dict) -> dict:
    stats: dict[str, list[float]] = {}
    top = d.get("stats") or {}
    for key in STAT_COLUMNS:
        if key in top:
            stats[key] = _stat_values(top[key])
    # Per-run stats take precedence when present (gives a real std over seeds).
    for key in STAT_COLUMNS:
        vals = [float(r["stats"][key]) for r in d.get("runs", [])
                if isinstance(r.get("stats"), dict) and isinstance(r["stats"].get(key), (int, float))]
        if vals:
            stats[key] = vals
    return {k: v for k, v in stats.items() if v}


def load_results(folder: Path) -> list[Result]:
    out = []
    for path in sorted(folder.glob("*.json")):  # glob on the folder itself: top level only
        if not path.is_file():
            continue
        try:
            d = json.loads(path.read_text())
        except (json.JSONDecodeError, OSError) as e:
            print(f"skip {path.name}: {e}")
            continue
        if d.get("eval_split") == "val":
            continue
        if not d.get("runs"):
            print(f"skip {path.name}: no runs")
            continue
        strategy = d.get("strategy", path.stem.split("_")[0])
        scenario = (d.get("config") or {}).get("scenario", "class")
        base = f"{strategy}_{scenario}"
        stem = path.stem
        tag = stem[len(base):].lstrip("_") if stem.startswith(base) else stem
        out.append(Result(stem, strategy, scenario, tag, d["runs"], collect_stats(d)))
    return out


def sort_key(r: Result):
    rank = MAIN_ORDER.index(r.strategy) if r.strategy in MAIN_ORDER else len(MAIN_ORDER)
    return (not r.is_main, rank, r.strategy, r.tag)


def pm(vals: np.ndarray, pct: bool = True, digits: int = 2) -> str:
    vals = np.asarray(vals, dtype=float)
    scale = 100.0 if pct else 1.0
    unit = "%" if pct else ""
    if len(vals) == 1:
        return f"{vals[0] * scale:.{digits}f}{unit}"
    return f"{vals.mean() * scale:.{digits}f} ± {vals.std() * scale:.{digits}f}{unit}"


def fmt_stat(vals: list[float]) -> str:
    a = np.array(vals, dtype=float)
    integral = np.allclose(a, np.round(a))
    if len(a) == 1:
        return f"{int(a[0]):,}" if integral else f"{a[0]:.2f}"
    digits = 0 if integral and a.mean() >= 100 else 1
    return f"{a.mean():,.{digits}f} ± {a.std():,.{digits}f}"


def method_label(r: Result) -> str:
    name = METHOD_NAME.get(r.strategy, r.strategy)
    return f"{name} [{r.tag}]" if r.tag else name


DATASET_TITLE = {"mnist": "Split MNIST", "cifar10": "Split CIFAR-10"}
TITLE = "Split MNIST"  # set in main() from the result files' "dataset" field


def dataset_title(folder: Path) -> str:
    """Name of the benchmark in `folder`. Files written before the --dataset flag existed
    have no "dataset" field; those are all MNIST."""
    names = set()
    for path in folder.glob("*.json"):
        try:
            names.add(json.loads(path.read_text()).get("dataset", "mnist"))
        except (ValueError, OSError):
            continue
    return " + ".join(DATASET_TITLE.get(n, n) for n in sorted(names)) or "Split MNIST"


def make_tables(results: list[Result]) -> str:
    scenarios = [s for s in SCENARIO_ORDER if any(r.scenario == s for r in results)]
    scenarios += sorted({r.scenario for r in results} - set(scenarios))
    parts = [f"# {TITLE} results (test split)", "",
             "Mean ± std over seeds. ACC = final average accuracy over all 5 tasks; "
             "forgetting = mean drop from best to final accuracy on tasks 1-4 "
             "(Chaudhry et al. 2018). Rows below the rule are tagged variants/ablations.", ""]
    for sc in scenarios:
        rows = sorted([r for r in results if r.scenario == sc], key=sort_key)
        stat_keys = [k for k in STAT_COLUMNS if any(k in r.stats for r in rows)]
        header = ["Method", "File", "ACC", "Forgetting", "Seeds"] + [STAT_COLUMNS[k] for k in stat_keys]
        parts += [f"## {SCENARIO_TITLE.get(sc, sc)} ({sc}-IL)", "",
                  "| " + " | ".join(header) + " |",
                  "|" + "|".join(["---"] * 2 + ["---:"] * (len(header) - 2)) + "|"]
        prev_main = None
        for r in rows:
            if prev_main is True and not r.is_main:
                parts.append("| " + " | ".join(["*variants*"] + [""] * (len(header) - 1)) + " |")
            prev_main = r.is_main
            cells = [method_label(r), f"`{r.label}`", pm(r.per_seed("final_avg_acc")),
                     pm(r.per_seed("forgetting")), str(len(r.runs))]
            cells += [fmt_stat(r.stats[k]) if k in r.stats else "–" for k in stat_keys]
            parts.append("| " + " | ".join(cells) + " |")
        parts.append("")
    return "\n".join(parts)


def color_for(strategy: str, extra: dict) -> str:
    if strategy in METHOD_COLOR:
        return METHOD_COLOR[strategy]
    if strategy not in extra:
        extra[strategy] = EXTRA_COLORS[len(extra) % len(EXTRA_COLORS)]
    return extra[strategy]


def main_methods(results: list[Result]) -> list[str]:
    present = {r.strategy for r in results if r.is_main}
    return [m for m in MAIN_ORDER if m in present] + sorted(present - set(MAIN_ORDER))


def present_scenarios(results: list[Result]) -> list[str]:
    """Scenarios that have main-method results, so no chart gets an empty panel."""
    return [s for s in SCENARIO_ORDER if any(r.is_main and r.scenario == s for r in results)] or ["class"]


def plot_avg_acc(results: list[Result], path: Path, extra: dict) -> None:
    scs = present_scenarios(results)
    fig, axes = plt.subplots(1, len(scs), figsize=(max(9.5, 5.5 * len(scs)), 4.4), squeeze=False)
    axes = axes[0]
    for ax, sc in zip(axes, scs):
        rows = {r.strategy: r for r in results if r.is_main and r.scenario == sc}
        lo = 100.0
        for m in main_methods(results):
            if m not in rows:
                continue
            c = rows[m].curves * 100
            x = np.arange(1, c.shape[1] + 1)
            mean, std = c.mean(0), c.std(0)
            lo = min(lo, float((mean - std).min()))
            col = color_for(m, extra)
            ax.plot(x, mean, marker="o", ms=4, lw=2, color=col,
                    label=f"{METHOD_NAME.get(m, m)} (n={len(c)})")
            ax.fill_between(x, mean - std, np.minimum(mean + std, 100), color=col, alpha=0.2, lw=0)
        ax.set_title(SCENARIO_TITLE[sc] + (" (primary)" if sc == "class" else ""))
        ax.set_xlabel("Tasks learned")
        ax.set_xticks(range(1, 6))
        # Zoom in when every curve stays high (task-IL), so differences are visible.
        ax.set_ylim(0 if lo < 60 else 10 * math.floor((lo - 1) / 10), 101)
        ax.yaxis.set_major_formatter(PercentFormatter(100))
        ax.grid(alpha=0.3)
        if not rows:
            ax.text(0.5, 0.5, "no results", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.legend(loc="lower left", fontsize=9, frameon=False)
    for ax in axes:
        ax.set_ylabel("Average accuracy on seen tasks")
    fig.suptitle(f"{TITLE}: average accuracy after each task (mean ± std over seeds)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_bars(results: list[Result], path: Path, extra: dict) -> None:
    methods = main_methods(results)
    metrics = [("final_avg_acc", "Final avg accuracy"), ("forgetting", "Forgetting")]
    scs = present_scenarios(results)
    fig, axes = plt.subplots(1, len(scs), figsize=(max(9.5, 5.5 * len(scs)), 4.4), sharey=True, squeeze=False)
    axes = axes[0]
    width = 0.8 / max(len(methods), 1)
    for ax, sc in zip(axes, scs):
        rows = {r.strategy: r for r in results if r.is_main and r.scenario == sc}
        for k, m in enumerate(methods):
            if m not in rows:
                continue
            vals = [rows[m].per_seed(key) * 100 for key, _ in metrics]
            means = [v.mean() for v in vals]
            stds = [v.std() for v in vals]
            x = np.arange(len(metrics)) + (k - (len(methods) - 1) / 2) * width
            bars = ax.bar(x, means, width * 0.92, yerr=stds, capsize=3, color=color_for(m, extra),
                          edgecolor="black", linewidth=0.5, label=METHOD_NAME.get(m, m),
                          error_kw={"elinewidth": 1})
            for b, mu, sd in zip(bars, means, stds):
                ax.text(b.get_x() + b.get_width() / 2, mu + sd + 1.5, f"{mu:.1f}",
                        ha="center", va="bottom", fontsize=7.5)
        ax.set_xticks(range(len(metrics)))
        ax.set_xticklabels([f"{lbl}\n({'higher' if key == 'final_avg_acc' else 'lower'} is better)"
                            for key, lbl in metrics])
        ax.set_title(SCENARIO_TITLE[sc] + (" (primary)" if sc == "class" else ""))
        ax.set_ylim(0, 112)
        ax.yaxis.set_major_formatter(PercentFormatter(100))
        ax.grid(axis="y", alpha=0.3)
        if not rows:
            ax.text(0.5, 0.5, "no results", ha="center", va="center", transform=ax.transAxes)
    handles, labels = [], []
    for ax in axes:
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                handles.append(h)
                labels.append(l)
    if labels:  # a folder with only tagged variants has no main methods to plot
        fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False,
                   bbox_to_anchor=(0.5, 0.95))
    fig.suptitle(f"{TITLE}: final accuracy and forgetting (mean ± std over seeds)")
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--results", default=str(HERE / "results"), help="folder with the result JSONs")
    p.add_argument("--out", default=None, help="output folder (default: same as --results)")
    args = p.parse_args()
    src = Path(args.results)
    out = Path(args.out) if args.out else src
    out.mkdir(parents=True, exist_ok=True)

    global TITLE
    TITLE = dataset_title(src)
    results = load_results(src)
    if not results:
        raise SystemExit(f"no test-split result files in {src}")
    md = make_tables(results)
    (out / "summary.md").write_text(md + "\n")
    print(md)
    extra: dict[str, str] = {}
    plot_avg_acc(results, out / "avg_acc.png", extra)
    plot_bars(results, out / "forgetting.png", extra)
    for name in ("summary.md", "avg_acc.png", "forgetting.png"):
        print(f"wrote {out / name}")


if __name__ == "__main__":
    main()

## Run the CIFAR-10 suite
Runs as a subprocess so its progress lines stream into the output below.

In [ ]:
import subprocess, sys
p = subprocess.Popen([sys.executable, "-u", "run_suite.py", "--only", "relax_cifar10"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("exit code", p.wait())

## Download the results
Download `cl_relax_cifar10_results.zip` (Kaggle: Output panel on the right; Colab: Files panel on the left) and unzip it inside the `continual_learning` folder on your computer.

In [ ]:
import shutil
dest = "/kaggle/working/cl_relax_cifar10_results" if os.path.isdir("/kaggle/working") else "/content/cl_relax_cifar10_results"
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.make_archive(dest, "zip", root_dir="/tmp/cl_project", base_dir="results")
print("saved", dest + ".zip")

## Report
Only drawn when this run produced test results (a tuning-only run may not).

In [ ]:
from IPython.display import Image, Markdown, display
import glob, subprocess, sys
if glob.glob("results/cifar10/*.json"):
    subprocess.run([sys.executable, "make_report.py", "--results", "results/cifar10"], check=False)
if os.path.exists("results/cifar10/summary.md"):
    display(Markdown(open("results/cifar10/summary.md").read()))
    for img in ("avg_acc.png", "forgetting.png"):
        if os.path.exists(f"results/cifar10/{img}"):
            display(Image(f"results/cifar10/{img}"))
else:
    print("No test results in this run, so no report. Tuning results are in the zip and the log above.")